# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their labels, and associated `@id` values in the dataset.

In [ ]:
# List all record sets (tables) in the dataset
print("Available record sets (by '@id' and name):")
record_set_ids = []
for rs in dataset.record_sets:
    print(f"- @id: {rs.id}, name: '{rs.name}'")
    record_set_ids.append(rs.id)

if record_set_ids:
    # For demonstration, show fields of the first record set
    first_rs = dataset.record_set(record_set_ids[0])
    print(f"\nFields for record set '@id': {first_rs.id} ('{first_rs.name}'):")
    for field in first_rs.fields:
        print(f"  - Field @id: {field.id}, column: {field.column_id}, dataType: {field.data_type}, label: '{field.name}'")
else:
    print("No record sets available in this dataset.")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, we'll extract all record sets into DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records with columns: {list(df.columns)}\n")
    else:
        print(f"No records found in record set: {record_set_id}\n")

# Select the main data record set for analysis (typically the main survey table)
if len(dataframes) > 0:
    # Choose the largest record set or the first one
    main_rs_id = max(dataframes, key=lambda x: len(dataframes[x]))
    print(f"Main analysis record set '@id': {main_rs_id}")
    print("Columns (fields) in this record set (by @id):")
    print(list(dataframes[main_rs_id].columns))
    display(dataframes[main_rs_id].head())
else:
    print("Did not load any record sets with data.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Below we'll select a numeric field (e.g., GAD-7 total score), filter records, normalize values, and group by a demographic field (e.g., gender), referencing all fields by their `@id`.

In [ ]:
# Identify the main DataFrame for EDA
df = dataframes[main_rs_id]

# Inspect available columns to select numeric and grouping fields (all are @id)
print(f"Available columns for EDA (field @ids):\n{df.columns.tolist()}")

# For demonstration, we'll pick some likely field ids (update as per actual @ids below)
# Example @id: 'gad7_total', 'gender', 'village_name', 'phq9_total', etc.
numeric_field_id = None
group_field_id = None
# Guess sensible field ids
prioritized_numeric_fields = [c for c in df.columns if 'gad' in c or 'phq' in c or 'psq' in c or 'score' in c or 'total' in c]
if prioritized_numeric_fields:
    numeric_field_id = prioritized_numeric_fields[0]
else:
    # Fallback to the first column if nothing matches
    numeric_field_id = df.columns[0]
print(f"Numeric field selected (by @id): {numeric_field_id}")

# Set group field as 'gender' if available, else 'village', else skip
for group in ['gender', 'village', 'village_name', 'level_of_education', 'marital_status']:
    for c in df.columns:
        if group in c:
            group_field_id = c
            break
    if group_field_id:
        break
print(f"Group field selected (by @id): {group_field_id}")

# Drop rows with missing numeric values and convert
filtered_df = df.copy()
filtered_df = filtered_df.dropna(subset=[numeric_field_id])
filtered_df = filtered_df[pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').notnull()]
filtered_df = filtered_df.astype({numeric_field_id: float})
# Example: filter to high values
threshold = filtered_df[numeric_field_id].mean() + filtered_df[numeric_field_id].std()
filtered_high = filtered_df[filtered_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean + 1 std): {len(filtered_high)} found\n")
display(filtered_high.head())

# Normalize field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by categorical field (e.g. gender) if available
if group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
    display(grouped_df)

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot the distribution of the selected numeric field, and if available, compare means across groups (e.g. gender).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field_id], kde=True, bins=15, color='skyblue')
plt.title(f"Distribution of '{numeric_field_id}' scores")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

if group_field_id is not None and group_field_id in filtered_df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"Comparison of '{numeric_field_id}' scores by '{group_field_id}'")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and analyze a FAIR² Croissant dataset using the `mlcroissant` library. Data was referenced throughout by entity `@id` to ensure precision and reproducibility.

- We surveyed the available record sets and fields (referenced by `@id`), extracted survey data, and performed filtering and normalization.
- Exploratory analysis was performed on a key numeric field, and basic group-wise comparisons were constructed.
- Data visualizations illustrated the distribution and group differences for survey metrics, facilitating mental health research insights.

Further work could address predictive modeling, richer visualizations, or domain-specific analyses using the FAIR² schema. For more information, consult the [mlcroissant documentation](https://github.com/mlcommons/croissant).